In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
guess_df = pd.read_csv('wordle-allowed-guesses.txt', header=None, names=['guess'])
allowed_df = pd.read_csv('wordle-allowed-guesses.txt', header=None, names=['allowed'])
past_df = pd.read_csv('past_answers.txt', header=None, names=['past'])

# parse past_df into several columns with format WORD PUZZLENUMBER DATE
past_df[['word', 'puzzle_number', 'date']] = past_df['past'].str.split(' ', expand=True)
# parse the date column into a datetime object that has format MM/DD/YY
# turn into string first
past_df['date'] = past_df['date'].astype(str)
past_df['date'] = past_df['date'].str.extract(r'(\d{1,2}/\d{1,2}/\d{2,4})')[0]
past_df['date'] = pd.to_datetime(past_df['date'], format='mixed')



past_df.head()

,past,word,puzzle_number,date
0,FORTH 1922 09/23/26,FORTH,1922,2026-09-23
1,STOKE 1921 09/22/26,STOKE,1921,2026-09-22
2,PAGER 1920 09/21/26,PAGER,1920,2026-09-21
3,SHONE 1919 09/20/26,SHONE,1919,2026-09-20
4,WAKEN 1918 09/19/26,WAKEN,1918,2026-09-19


In [22]:
allowed_df.head()

,allowed
0,aahed
1,aalii
2,aargh
3,aarti
4,abaca


In [23]:
# uniform all the words in guess_df, allowed_df, and past_df to be lowercase
past_df['word'] = past_df['word'].str.lower()
past_df.head()

,past,word,puzzle_number,date
0,FORTH 1922 09/23/26,forth,1922,2026-09-23
1,STOKE 1921 09/22/26,stoke,1921,2026-09-22
2,PAGER 1920 09/21/26,pager,1920,2026-09-21
3,SHONE 1919 09/20/26,shone,1919,2026-09-20
4,WAKEN 1918 09/19/26,waken,1918,2026-09-19


In [24]:
# find how many word in past_df are in guess_df and allowed_df
past_in_guess = past_df[past_df['word'].isin(guess_df['guess'])]
past_in_allowed = past_df[past_df['word'].isin(allowed_df['allowed'])]

print(f"Number of past words in guess_df: {len(past_in_guess)}")
print(f"Number of past words in allowed_df: {len(past_in_allowed)}")


Number of past words in guess_df: 60
Number of past words in allowed_df: 60


In [25]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

def load_and_clean_history(filepath):
    records = []
    # Using 2026-09-23 as the reference date based on the latest dataset entry
    reference_date = datetime(2026, 9, 23) 

    with open(filepath, 'r') as f:
        for line in f:
            # Address trailing character noise: Strip non-alphanumeric trailing artifacts like '@'
            line = re.sub(r'[^a-zA-Z0-9\s/]', '', line.strip())
            parts = line.split()
            
            if len(parts) >= 3:
                # Casing inconsistency: Standardize all strings to lowercase
                word = parts[0].lower() 
                # Split puzzle indexing: Store as string to safely capture IDs like "325a"
                puzzle_id = str(parts[1]) 
                date_str = parts[2]
                
                # Parse the date to calculate recency decay for the Bayesian prior
                date_obj = datetime.strptime(date_str, "%m/%d/%y")
                days_since = (reference_date - date_obj).days
                
                records.append({
                    'word': word,
                    'puzzle_id': puzzle_id,
                    'days_since': days_since
                })
                
    return pd.DataFrame(records)

def load_wordle_dictionaries(answers_path, guesses_path):
    with open(answers_path, 'r') as f:
        answers = set(word.strip().lower() for word in f.readlines())
        
    with open(guesses_path, 'r') as f:
        guesses = set(word.strip().lower() for word in f.readlines())
        
    # Ensure set integrity: Answers must be a subset of allowed guesses
    missing_guesses = answers - guesses
    if missing_guesses:
        print(f"Warning: {len(missing_guesses)} answers are missing from the guesses list.")
        
    return list(answers), list(guesses)

def compute_hybrid_priors(df_history, valid_answers, df_kaggle_unigram, alpha=0.5):
    # Initialize dataframe with the exact 2,315 valid solutions
    df_priors = pd.DataFrame({'word': valid_answers})
    
    # Format and merge the Kaggle Corpus frequency counts
    df_kaggle_unigram['word'] = df_kaggle_unigram['word'].astype(str).str.lower()
    df_priors = df_priors.merge(df_kaggle_unigram[['word', 'count']], on='word', how='left')
    
    # Feature 2: Smoothed Corpus Log-Frequency
    # Fill missing unigram frequencies with 1 (Laplace smoothing)
    df_priors['count'] = df_priors['count'].fillna(1)
    df_priors['S_word'] = np.log10(1 + df_priors['count'])
    df_priors['P_corpus'] = df_priors['S_word'] / df_priors['S_word'].sum()
    
    # Merge historical data to evaluate recency
    df_priors = df_priors.merge(df_history[['word', 'days_since']], on='word', how='left')
    
    # Feature 1: Recency Penalty
    def apply_historical_penalty(days):
        if pd.isna(days):
            return 1.0   # Never appeared: maximum historical probability weight
        elif days <= 180:
            return 0.01  # Appeared recently: extremely unlikely to repeat (gamma)
        else:
            return 0.10  # Appeared > 180 days ago: possible but penalized (delta)

    df_priors['P_history'] = df_priors['days_since'].apply(apply_historical_penalty)
    
    # Feature 4: Hybrid Bayesian Prior calculation
    df_priors['P_prior'] = (alpha * df_priors['P_corpus']) + ((1 - alpha) * df_priors['P_history'])
    
    # Normalize the final hybrid probabilities so they sum to 1.0 for the entropy calculation
    df_priors['P_prior'] = df_priors['P_prior'] / df_priors['P_prior'].sum()
    
    # Clean up intermediate calculation columns
    df_priors = df_priors[['word', 'P_prior', 'count', 'days_since']]
    
    return df_priors.sort_values(by='P_prior', ascending=False)

if __name__ == "__main__":
    # 1. Execute cleaning pipelines
    df_history = load_and_clean_history('past_answers.txt')
    answers, guesses = load_wordle_dictionaries('wordle-answers-alphabetical.txt', 'wordle-allowed-guesses.txt')
    
    # 2. Ingest external dataset (Load your unigram_freq.csv here)
    try:
        df_kaggle = pd.read_csv('unigram_freq.csv')
    except FileNotFoundError:
        # Fallback dummy data if CSV is not locally present during initial test
        print("Kaggle unigram dataset not found. Using fallback data.")
        df_kaggle = pd.DataFrame({
            'word': ['crane', 'salet', 'xylyl', 'forth'], 
            'count': [5400000, 15000, 10, 320000]
        })
    
    # 3. Generate the finalized Prior distribution table
    df_final_priors = compute_hybrid_priors(df_history, answers, df_kaggle)
    print("Top 10 candidate words by prior probability:")
    print(df_final_priors.head(10))

Kaggle unigram dataset not found. Using fallback data.
Top 10 candidate words by prior probability:
       word   P_prior  count  days_since
623   truce  0.001547    1.0         NaN
1946  aunty  0.001547    1.0         NaN
1496  fudge  0.001547    1.0         NaN
970   bison  0.001547    1.0         NaN
1941  bowel  0.001547    1.0         NaN
968   tramp  0.001547    1.0         NaN
1939  spawn  0.001547    1.0         NaN
966   fight  0.001547    1.0         NaN
965   piggy  0.001547    1.0         NaN
1501  twixt  0.001547    1.0         NaN


In [ ]:
# see the remaining words in the candidate pool that are not in the past answers
remaining_words = [word for word in candidates if word not in past_answers]


In [41]:
"""
Wordle Solver — Data Cleaning & Exploratory Data Analysis
==========================================================
Inputs (raw, as provided):
    - past_answers.txt              "WORD  PUZZLE#  MM/DD/YY[@]"  (NYT history)
    - wordle-allowed-guesses.txt    one 5-letter word per line (valid guesses only,
                                     i.e. NOT including the words in the answer list)
    - wordle-answers-alphabetical.txt  one 5-letter word per line (possible secret words)

Why this cleaning matters for the algorithm plan:
    - The solver's belief state (Lecture 3/5: candidate set + prior) should be built
      over the words that can STILL be picked as a future secret, not the full
      historical answer list. NYT does not repeat an answer once it has been used,
      so `past_answers` must be subtracted from the answer pool before any entropy /
      Bayesian calculation — otherwise the solver spends guesses ruling out words
      that were already used in 2021-2026 and can never be the target again.
    - The "allowed guesses" list is the full legal-guess vocabulary (10,656 words)
      used to search for high-information guesses; it is a superset check target —
      every answer must also be a legal guess.

Outputs (written to ./cleaned/):
    - remaining_candidates.txt   the real, future-facing secret-word pool
    - past_answers_clean.csv     parsed, typed history (word, puzzle_id, date)
    - full_guess_vocab.txt       union of allowed guesses + all possible answers
    - letter_freq_overall.csv, letter_freq_positional.csv
    - figures: length_check.png, letter_freq.png, positional_heatmap.png,
               history_timeline.png, repeated_letters.png
    - eda_summary.txt            plain-text findings to quote in the report
"""

import re
import string
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW_DIR = Path("C:\\Users\\ASUS\\projects\\wordleSolver")
OUT_DIR = Path("C:\\Users\\ASUS\\projects\\wordleSolver\\cleaned")
FIG_DIR = Path("C:\\Users\\ASUS\\projects\\wordleSolver\\figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

WORD_RE = re.compile(r"^[a-z]{5}$")

summary_lines = []


def log(msg=""):
    """Print + collect for the eda_summary.txt report."""
    print(msg)
    summary_lines.append(msg)


# ---------------------------------------------------------------------------
# 1. LOAD + CLEAN each raw file
# ---------------------------------------------------------------------------

def load_wordlist(path: Path) -> tuple[list[str], list[str]]:
    """Lowercase, strip, dedupe, validate 5-letter alphabetic words.
    Returns (clean_words, rejected_raw_lines)."""
    clean, rejected = [], []
    seen = set()
    for raw in path.read_text().splitlines():
        w = raw.strip().lower()
        if not w:
            continue
        if not WORD_RE.match(w):
            rejected.append(raw)
            continue
        if w in seen:
            rejected.append(f"{raw}  <-- duplicate")
            continue
        seen.add(w)
        clean.append(w)
    return clean, rejected


def load_past_answers(path: Path) -> pd.DataFrame:
    """Parse 'WORD  PUZZLE#  MM/DD/YY[@]' into a typed DataFrame."""
    rows, rejected = [], []
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 3:
            rejected.append(raw)
            continue
        word, puzzle_id, date_str = parts
        word = word.lower()
        has_marker = date_str.endswith("@")
        date_str = date_str.rstrip("@")
        if not WORD_RE.match(word):
            rejected.append(raw)
            continue
        try:
            date = datetime.strptime(date_str, "%m/%d/%y")
        except ValueError:
            rejected.append(raw)
            continue
        # A few early puzzle IDs carry an 'a'/'b' suffix (e.g. "325a", "325b"):
        # these are the documented cases where NYT swapped that day's answer for a
        # different word shortly after acquiring the game in 2022, leaving two
        # words attached to the same calendar date. Keep the numeric ID plus the
        # variant suffix separately rather than dropping or silently coercing it.
        m = re.match(r"^(\d+)([a-z]?)$", puzzle_id)
        if not m:
            rejected.append(raw)
            continue
        numeric_id, variant_suffix = int(m.group(1)), m.group(2)
        rows.append(
            {
                "word": word,
                "puzzle_id": numeric_id,
                "variant": variant_suffix,  # '' normally, 'a'/'b' for swapped-answer days
                "date": date,
                "flagged": has_marker,  # meaning undocumented upstream; kept, not dropped
            }
        )
    if rejected:
        log(f"[past_answers] rejected {len(rejected)} malformed line(s): {rejected[:5]}")
    # Drop only exact duplicate rows (same word+puzzle_id+date repeated in the raw file).
    # Do NOT dedupe by word alone: a small number of words genuinely appear on two
    # different dates in this history (see repeated-word diagnostic below), and
    # collapsing those would silently delete a real, distinct puzzle-day record.
    df = pd.DataFrame(rows).drop_duplicates(subset=["word", "puzzle_id", "date"])
    df = df.sort_values(["puzzle_id", "variant"]).reset_index(drop=True)
    return df


answers_all, answers_rejected = load_wordlist(RAW_DIR / "wordle-answers-alphabetical.txt")
guesses_only, guesses_rejected = load_wordlist(RAW_DIR / "wordle-allowed-guesses.txt")
past_df = load_past_answers(RAW_DIR / "past_answers.txt")

log("=" * 70)
log("1. RAW LOAD SUMMARY")
log("=" * 70)
log(f"answer list (wordle-answers-alphabetical.txt): {len(answers_all)} clean words, "
    f"{len(answers_rejected)} rejected")
log(f"allowed-guesses-only list (wordle-allowed-guesses.txt): {len(guesses_only)} clean words, "
    f"{len(guesses_rejected)} rejected")
log(f"past_answers.txt: {len(past_df)} clean rows")
if guesses_rejected:
    log(f"  sample rejected guesses: {guesses_rejected[:5]}")

# ---------------------------------------------------------------------------
# 2. CROSS-LIST CONSISTENCY CHECKS
# ---------------------------------------------------------------------------
log()
log("=" * 70)
log("2. CROSS-LIST CONSISTENCY CHECKS")
log("=" * 70)

answers_set = set(answers_all)
guesses_only_set = set(guesses_only)
overlap = answers_set & guesses_only_set
log(f"Overlap between 'answers' and 'allowed-guesses-only' lists: {len(overlap)} words "
    f"(expected 0 if the two files are meant to be disjoint partitions)")

full_guess_vocab = sorted(answers_set | guesses_only_set)
log(f"Full legal guess vocabulary (union): {len(full_guess_vocab)} words")

past_set = set(past_df["word"])
past_not_in_answers = past_set - answers_set
log(f"Past answers NOT found in the current answer list: {len(past_not_in_answers)} "
    f"{sorted(past_not_in_answers)[:10] if past_not_in_answers else ''}")
log("  (>0 is expected/known: NYT has retired a handful of answers, e.g. politically "
    "or culturally sensitive words, since taking over the game in 2022.)")

# ---------------------------------------------------------------------------
# 3. DERIVE THE REAL, FUTURE-FACING CANDIDATE POOL
#    -> this is the actual belief-state universe the solver should reason over
# ---------------------------------------------------------------------------
remaining_candidates = sorted(answers_set - past_set)

log()
log("=" * 70)
log("3. REMAINING CANDIDATE POOL (the number that actually matters)")
log("=" * 70)
log(f"Total possible answers ever published as a list:  {len(answers_set)}")
log(f"Already used as of the most recent puzzle in the data: {len(past_set & answers_set)}")
log(f"REMAINING CANDIDATES for a future puzzle:          {len(remaining_candidates)}")
log(f"Uniform-prior starting entropy over full list:      {np.log2(len(answers_set)):.3f} bits")
log(f"Uniform-prior starting entropy over remaining pool: {np.log2(len(remaining_candidates)):.3f} bits")
log("  -> Building the solver on the full 2,3xx-word list instead of the remaining pool "
    "silently overstates the problem's difficulty and is the wrong ground truth for a "
    "'genuine real-world' claim in the report.")

# ---------------------------------------------------------------------------
# 4. STRUCTURAL VALIDATION (length, alphabet, duplicate letters)
# ---------------------------------------------------------------------------
log()
log("=" * 70)
log("4. STRUCTURAL CHECKS")
log("=" * 70)
lengths = {len(w) for w in full_guess_vocab}
log(f"Distinct word lengths present in full vocab: {lengths} (must be {{5}})")

dup_letter_counts = Counter(len(w) - len(set(w)) for w in remaining_candidates)
log(f"Repeated-letter distribution in remaining candidates "
    f"(0 = all-unique-letters word): {dict(sorted(dup_letter_counts.items()))}")
words_with_repeats = sum(v for k, v in dup_letter_counts.items() if k > 0)
log(f"-> {words_with_repeats}/{len(remaining_candidates)} remaining candidates "
    f"({words_with_repeats/len(remaining_candidates):.1%}) contain a repeated letter — "
    f"relevant because a naive feedback-pattern encoder that ignores duplicate-letter "
    f"tile-coloring rules will silently mis-score these.")

# ---------------------------------------------------------------------------
# 5. LETTER FREQUENCY — overall and positional (feeds the entropy heuristic)
# ---------------------------------------------------------------------------
log()
log("=" * 70)
log("5. LETTER FREQUENCY (computed on the REMAINING candidate pool)")
log("=" * 70)

overall_counts = Counter("".join(remaining_candidates))
overall_freq = pd.Series(overall_counts).sort_values(ascending=False)
overall_freq.to_csv(OUT_DIR / "letter_freq_overall.csv", header=["count"])
log("Top 10 most common letters (remaining pool):")
log(overall_freq.head(10).to_string())

pos_counts = {i: Counter(w[i] for w in remaining_candidates) for i in range(5)}
pos_df = pd.DataFrame(pos_counts).fillna(0).astype(int)
pos_df.index.name = "letter"
pos_df.columns = [f"pos_{i+1}" for i in range(5)]
pos_df.to_csv(OUT_DIR / "letter_freq_positional.csv")
log()
log("Most common letter at each position:")
for c in pos_df.columns:
    top_letter = pos_df[c].idxmax()
    log(f"  {c}: '{top_letter}' ({pos_df[c].max()} occurrences)")

vowel_set = set("aeiou")
vowel_counts = Counter(sum(ch in vowel_set for ch in w) for w in remaining_candidates)
log(f"\nVowel-count distribution per word: {dict(sorted(vowel_counts.items()))}")

# ---------------------------------------------------------------------------
# 6. HISTORY / TIME-SERIES SANITY CHECKS on past_answers
# ---------------------------------------------------------------------------
log()
log("=" * 70)
log("6. PAST-ANSWERS HISTORY CHECKS")
log("=" * 70)
past_df_sorted = past_df.sort_values("puzzle_id").reset_index(drop=True)
expected_ids = set(range(past_df_sorted["puzzle_id"].min(), past_df_sorted["puzzle_id"].max() + 1))
actual_ids = set(past_df_sorted["puzzle_id"])
missing_ids = sorted(expected_ids - actual_ids)
log(f"Puzzle-ID range: {past_df_sorted['puzzle_id'].min()}–{past_df_sorted['puzzle_id'].max()} "
    f"({len(expected_ids)} expected daily puzzles)")
log(f"Missing puzzle IDs in the data: {len(missing_ids)} {missing_ids[:10]}")
date_gaps = past_df_sorted["date"].diff().dt.days.dropna()
log(f"Date gap between consecutive rows — should be 1.0 day if no rows are missing: "
    f"min={date_gaps.min()}, max={date_gaps.max()}, mean={date_gaps.mean():.2f}")
log(f"Rows with the undocumented trailing '@' marker: {past_df_sorted['flagged'].sum()} "
    f"(kept as-is — worth footnoting in the report as an unexplained upstream artifact "
    f"rather than silently dropping them)")

variant_rows = past_df_sorted[past_df_sorted["variant"] != ""]
if not variant_rows.empty:
    log(f"\nSwapped-answer days (two words sharing one puzzle ID, 'a'/'b' suffix): "
        f"{variant_rows['puzzle_id'].nunique()} puzzle day(s)")
    for pid, grp in variant_rows.groupby("puzzle_id"):
        words = ", ".join(f"{r.word}({r.variant})" for r in grp.itertuples())
        log(f"  puzzle #{pid} on {grp['date'].iloc[0].date()}: {words}")
    log("  -> Documented case of NYT replacing an already-scheduled answer shortly after "
        "acquiring the game in 2022. Both words should count as 'used' when building the "
        "remaining-candidate pool, which the set-difference above already does correctly.")

word_counts = past_df_sorted["word"].value_counts()
reused_words = word_counts[word_counts > 1]
if not reused_words.empty:
    log(f"\nWords that appear on more than one distinct date in this history: "
        f"{len(reused_words)} {list(reused_words.index[:10])}")
    log("  -> Worth a sentence in the report: either the source list mixes the pre-NYT "
        "and NYT eras with an accidental repeat, or these are genuine reruns. Either way, "
        "for the remaining-candidate pool a word only needs to be excluded once, so this "
        "doesn't change the set-difference result — but it's a fact worth citing since a "
        "reader may otherwise assume the history has zero repeats.")

# ---------------------------------------------------------------------------
# 7. SAVE CLEANED ARTIFACTS
# ---------------------------------------------------------------------------
(OUT_DIR / "remaining_candidates.txt").write_text("\n".join(remaining_candidates))
(OUT_DIR / "full_guess_vocab.txt").write_text("\n".join(full_guess_vocab))
past_df_sorted.to_csv(OUT_DIR / "past_answers_clean.csv", index=False)

log()
log("=" * 70)
log("7. FILES WRITTEN")
log("=" * 70)
for f in sorted(OUT_DIR.iterdir()):
    log(f"  {f}")

# ---------------------------------------------------------------------------
# 8. FIGURES
# ---------------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

# 8a. Length / validity check
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["answers", "allowed-guesses", "remaining"],
       [len(answers_all), len(guesses_only), len(remaining_candidates)],
       color=["#4C72B0", "#55A868", "#C44E52"])
ax.set_ylabel("word count")
ax.set_title("Word-list sizes after cleaning")
for i, v in enumerate([len(answers_all), len(guesses_only), len(remaining_candidates)]):
    ax.text(i, v + 50, str(v), ha="center", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "list_sizes.png")
plt.close(fig)

# 8b. Overall letter frequency
fig, ax = plt.subplots(figsize=(8, 3))
overall_freq.reindex(list(string.ascii_lowercase)).fillna(0).plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Overall letter frequency — remaining candidate pool")
ax.set_ylabel("occurrences")
fig.tight_layout()
fig.savefig(FIG_DIR / "letter_freq.png")
plt.close(fig)

# 8c. Positional heatmap
fig, ax = plt.subplots(figsize=(5, 8))
im = ax.imshow(pos_df.reindex(list(string.ascii_lowercase)).fillna(0).values, aspect="auto", cmap="viridis")
ax.set_yticks(range(26))
ax.set_yticklabels(list(string.ascii_lowercase))
ax.set_xticks(range(5))
ax.set_xticklabels(pos_df.columns)
ax.set_title("Letter frequency by position")
fig.colorbar(im, ax=ax, shrink=0.6, label="count")
fig.tight_layout()
fig.savefig(FIG_DIR / "positional_heatmap.png")
plt.close(fig)

# 8d. History timeline: cumulative answers used vs. remaining pool shrinking over time
fig, ax = plt.subplots(figsize=(7, 3.5))
cum_used = np.arange(1, len(past_df_sorted) + 1)
remaining_over_time = len(answers_set) - cum_used
ax.plot(past_df_sorted["date"], remaining_over_time, color="#C44E52")
ax.set_title("Remaining candidate pool size over time")
ax.set_ylabel("words left in the answer pool")
ax.set_xlabel("date")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIG_DIR / "history_timeline.png")
plt.close(fig)

# 8e. Repeated-letter distribution
fig, ax = plt.subplots(figsize=(4, 3))
keys = sorted(dup_letter_counts)
ax.bar([str(k) for k in keys], [dup_letter_counts[k] for k in keys], color="#8172B2")
ax.set_xlabel("number of repeated letters in the word")
ax.set_ylabel("word count")
ax.set_title("Repeated-letter distribution (remaining pool)")
fig.tight_layout()
fig.savefig(FIG_DIR / "repeated_letters.png")
plt.close(fig)

log()
log("Figures written to: " + str(FIG_DIR))

(OUT_DIR / "eda_summary.txt").write_text("\n".join(summary_lines))
print("\nDone.")

1. RAW LOAD SUMMARY
answer list (wordle-answers-alphabetical.txt): 2315 clean words, 0 rejected
allowed-guesses-only list (wordle-allowed-guesses.txt): 10657 clean words, 0 rejected
past_answers.txt: 1927 clean rows

2. CROSS-LIST CONSISTENCY CHECKS
Overlap between 'answers' and 'allowed-guesses-only' lists: 0 words (expected 0 if the two files are meant to be disjoint partitions)
Full legal guess vocabulary (union): 12972 words
Past answers NOT found in the current answer list: 60 ['aloha', 'aspic', 'aster', 'atlas', 'atria', 'balsa', 'beaut', 'beige', 'capon', 'carom']
  (>0 is expected/known: NYT has retired a handful of answers, e.g. politically or culturally sensitive words, since taking over the game in 2022.)

3. REMAINING CANDIDATE POOL (the number that actually matters)
Total possible answers ever published as a list:  2315
Already used as of the most recent puzzle in the data: 1842
REMAINING CANDIDATES for a future puzzle:          473
Uniform-prior starting entropy over full

In [46]:
# see if remaining words in the candidate pool have repeated letters
repeated_letter_words = pd.read_csv('cleaned/remaining_candidates.txt', header=None, names=['word'])
repeated_letter_words = [word for word in repeated_letter_words['word'] if len(set(word)) < len(word)]
repeated_letter_counts = Counter(len(word) - len(set(word)) for word in repeated_letter_words)

print(f"Words with repeated letters: {len(repeated_letter_words)}")



Words with repeated letters: 179


In [ ]:
# see if there are any repeated words in the past answers
repeated_words = past_df['word'].value_counts()
print(f"Repeated words in past answers: {len(repeated_words[repeated_words > 1])}")
# see what the repeated words are and how many times they appear
repeated_words_list = repeated_words[repeated_words > 1]
print(f"Repeated words and their counts:\n{repeated_words_list}")
# print only the words


Repeated words in past answers: 25
Repeated words and their counts:
word
cigar    2
stand    2
clock    2
sandy    2
squad    2
break    2
linen    2
trace    2
forth    2
ivory    2
feign    2
baton    2
stout    2
gripe    2
click    2
flume    2
agree    2
acute    2
thumb    2
bring    2
comet    2
grade    2
awake    2
focus    2
smile    2
Name: count, dtype: int64
Repeated words and their counts:
word
cigar    2
stand    2
clock    2
sandy    2
squad    2
break    2
linen    2
trace    2
forth    2
ivory    2
feign    2
baton    2
stout    2
gripe    2
click    2
flume    2
agree    2
acute    2
thumb    2
bring    2
comet    2
grade    2
awake    2
focus    2
smile    2
Name: count, dtype: int64


In [38]:
from wordle_engine import *

"""
demo_benchmark.py
================================================================================
Usage example for wordle_engine.py — copy these cells into analysis.ipynb.

Run as a script (`python demo_benchmark.py`) or paste cell-by-cell into the
notebook; nothing here depends on being run as __main__.
"""

import time
import pandas as pd

from wordle_engine import (
    WordleEngine, load_word_pools,
    BaselineGreedyEntropySolver, MultiStepLookaheadSolver,
    InformationMetricsMatrix, play_game, simulate_all, summarize,
)

# ------------------------------------------------------------------ #
# 1. Load data (prefers ./cleaned/, falls back to the raw .txt files) #
# ------------------------------------------------------------------ #
candidates, guess_vocab = load_word_pools(".")
engine = WordleEngine(candidates, guess_vocab)
print(f"remaining candidate pool: {len(candidates)} words")
print(f"full legal guess vocabulary: {len(guess_vocab)} words")

# ------------------------------------------------------------------ #
# 2. Pick the opening word once, scanning the FULL guess vocabulary. #
#    This is the single most expensive entropy scan (~3s) — do it    #
#    once and reuse the result everywhere else instead of re-scanning#
#    12,972 words on every game in the benchmark loop.                #
# ------------------------------------------------------------------ #
t0 = time.time()
opener_finder = BaselineGreedyEntropySolver(engine, guess_pool=guess_vocab)
OPENING_WORD = opener_finder.choose(candidates)
print(f"opening word (full-vocab entropy max): '{OPENING_WORD}' "
      f"({time.time()-t0:.2f}s to compute)")

# ------------------------------------------------------------------ #
# 3. Information Metrics Matrix — 3Blue1Brown-style comparison table  #
#    for a shortlist of well-known candidate openers, PLUS the actual #
#    information received once a specific secret is revealed.         #
# ------------------------------------------------------------------ #
imm = InformationMetricsMatrix(engine)
demo_secret = candidates[3]
shortlist = sorted({OPENING_WORD, "crane", "slate", "adieu", "roate", "trace", "spare"})
metrics_table = imm.table(candidates, shortlist, secret=demo_secret)
print(f"\nInformation Metrics Matrix (secret used for 'actual' columns: '{demo_secret}')")
print(metrics_table.to_string(index=False))

# ------------------------------------------------------------------ #
# 4. Single game trace, so you can see the candidate set narrow turn  #
#    by turn (useful for the report's worked example / screenshots).  #
# ------------------------------------------------------------------ #
fast_pool_solver = BaselineGreedyEntropySolver(engine, guess_pool=candidates)
trace = play_game(engine, fast_pool_solver, secret=demo_secret, first_guess=OPENING_WORD)
print(f"\nWorked example — secret='{demo_secret}':")
for i, (g, p) in enumerate(zip(trace["guesses"], trace["patterns"]), 1):
    print(f"  guess {i}: {g}  ->  {p}")
print(f"  solved in {trace['n_guesses']} guesses" if trace["solved"] else "  NOT solved within limit")

# ------------------------------------------------------------------ #
# 5. Headless benchmark: play EVERY word in the remaining candidate   #
#    pool as the secret, for each solver. This is the number to       #
#    quote in the report and compare against the published 3.4212     #
#    optimum (computed over the larger, 2,315-word full answer list). #
#    ~25-30s each on this machine for 473 secrets.                    #
# ------------------------------------------------------------------ #
solvers = {
    "baseline": BaselineGreedyEntropySolver(engine, guess_pool=candidates),
    "lookahead": MultiStepLookaheadSolver(engine, guess_pool=candidates, top_k=8, exact_threshold=12),
}

results = []
for label, solver in solvers.items():
    t0 = time.time()
    df = simulate_all(engine, solver, first_guess=OPENING_WORD, verbose_every=0)
    elapsed = time.time() - t0
    stats = summarize(df, solver.name)
    stats["seconds"] = round(elapsed, 1)
    results.append(stats)
    print(f"\n{solver.name}: mean={stats['mean_guesses']}  "
          f"median={stats['median_guesses']}  max={stats['max_guesses']}  "
          f"({elapsed:.1f}s for {len(df)} games)")
    print(f"  distribution: {dict(sorted(stats['distribution'].items()))}")

comparison = pd.DataFrame(
    [{k: v for k, v in r.items() if k != "distribution"} for r in results]
)
print("\nSolver comparison summary:")
print(comparison.to_string(index=False))
comparison.to_csv("cleaned/solver_comparison.csv", index=False)

# ------------------------------------------------------------------ #
# Reference point for the report:
#   Selby (2022) / Bertsimas & Paskov (2024): 3.4212 average guesses,
#   exhaustive search, opening word "salet", over the FULL 2,315-word
#   answer list. Our numbers above are computed over the smaller
#   473-word REMAINING pool (already-used NYT answers excluded), which
#   is why they land below 3.42 — it's a genuinely easier instance of
#   the same problem, not an apples-to-apples beat of the optimum.
# ------------------------------------------------------------------ #




remaining candidate pool: 473 words
full legal guess vocabulary: 12972 words
opening word (full-vocab entropy max): 'saner' (4.74s to compute)

Information Metrics Matrix (secret used for 'actual' columns: 'abuse')
guess  prior_entropy_bits  expected_info_bits  n_buckets  max_bucket_frac actual_pattern  actual_info_bits  candidates_remaining
saner               8.886               5.668         94            0.129          YY.Y.             8.886                     1
slate               8.886               5.462         91            0.110          Y.Y.G             7.301                     3
roate               8.886               5.403         81            0.129          ..Y.G             5.716                     9
trace               8.886               5.343         84            0.154          ..Y.G             5.716                     9
crane               8.886               5.315         85            0.169          ..Y.G             5.716                     9
spare      

In [39]:
"""
demo_benchmark.py
================================================================================
Usage example for wordle_engine.py — copy these cells into analysis.ipynb.

"""

import time
import pandas as pd

from wordle_engine import (
    WordleEngine, load_word_pools,
    BaselineGreedyEntropySolver, MultiStepLookaheadSolver,
    InformationMetricsMatrix, play_game, simulate_all, summarize,
)

# ------------------------------------------------------------------ #
# 1. Load data (prefers ./cleaned/, falls back to the raw .txt files) #
# ------------------------------------------------------------------ #
candidates, guess_vocab = load_word_pools(".")
engine = WordleEngine(candidates, guess_vocab)
print(f"remaining candidate pool: {len(candidates)} words")
print(f"full legal guess vocabulary: {len(guess_vocab)} words")


# 2. Pick the opening word once, scanning the FULL guess vocabulary. #
#    This is the single most expensive entropy scan (~3s) — do it    #
#    once and reuse the result everywhere else instead of re-scanning#
#    12,972 words on every game in the benchmark loop.                #

t0 = time.time()
opener_finder = BaselineGreedyEntropySolver(engine, guess_pool=guess_vocab)
OPENING_WORD = opener_finder.choose(candidates)
print(f"opening word (full-vocab entropy max): '{OPENING_WORD}' "
      f"({time.time()-t0:.2f}s to compute)")


# 3. Information Metrics Matrix — 3Blue1Brown-style reference
#    for a shortlist of well-known candidate openers, PLUS the actual #
#    information received once a specific secret is revealed.         #

imm = InformationMetricsMatrix(engine)
demo_secret = candidates[0]
shortlist = sorted({OPENING_WORD, "crane", "slate", "adieu", "roate", "trace", "spare"}) # shortlist of candidate openers
metrics_table = imm.table(candidates, shortlist, secret=demo_secret)
print(f"\nInformation Metrics Matrix (secret used for 'actual' columns: '{demo_secret}')")
print(metrics_table.to_string(index=False))


# 4. Single game trace, so you can see the candidate set narrow turn  #
#    by turn (useful for the report's worked example / screenshots).  #

fast_pool_solver = BaselineGreedyEntropySolver(engine, guess_pool=candidates)
trace = play_game(engine, fast_pool_solver, secret=demo_secret, first_guess=OPENING_WORD)
print(f"\nWorked example — secret='{demo_secret}':")
for i, (g, p) in enumerate(zip(trace["guesses"], trace["patterns"]), 1):
    print(f"  guess {i}: {g}  ->  {p}")
print(f"  solved in {trace['n_guesses']} guesses" if trace["solved"] else "  NOT solved within limit")

# ------------------------------------------------------------------ #
# 5. Headless benchmark: play EVERY word in the remaining candidate   #
#    pool as the secret, for each solver. This is the number to       #
#    quote in the report and compare against the published 3.4212     #
#    optimum (computed over the larger, 2,315-word full answer list). #
#    ~25-30s each on this machine for 473 secrets.                    #
# ------------------------------------------------------------------ #
solvers = {
    "baseline": BaselineGreedyEntropySolver(engine, guess_pool=candidates),
    "lookahead": MultiStepLookaheadSolver(engine, guess_pool=candidates, top_k=8, exact_threshold=12),
}

results = []
for label, solver in solvers.items():
    t0 = time.time()
    df = simulate_all(engine, solver, first_guess=OPENING_WORD, verbose_every=0)
    elapsed = time.time() - t0
    stats = summarize(df, solver.name)
    stats["seconds"] = round(elapsed, 1)
    results.append(stats)
    print(f"\n{solver.name}: mean={stats['mean_guesses']}  "
          f"median={stats['median_guesses']}  max={stats['max_guesses']}  "
          f"({elapsed:.1f}s for {len(df)} games)")
    print(f"  distribution: {dict(sorted(stats['distribution'].items()))}")

comparison = pd.DataFrame(
    [{k: v for k, v in r.items() if k != "distribution"} for r in results]
)
print("\nSolver comparison summary:")
print(comparison.to_string(index=False))
comparison.to_csv("cleaned/solver_comparison.csv", index=False)

# ------------------------------------------------------------------ #
# Reference point for the report:
#   Selby (2022) / Bertsimas & Paskov (2024): 3.4212 average guesses,
#   exhaustive search, opening word "salet", over the FULL 2,315-word
#   answer list. Our numbers above are computed over the smaller
#   473-word REMAINING pool (already-used NYT answers excluded), which
#   is why they land below 3.42 — it's a genuinely easier instance of
#   the same problem, not an apples-to-apples beat of the optimum.
# ------------------------------------------------------------------ #


remaining candidate pool: 473 words
full legal guess vocabulary: 12972 words
opening word (full-vocab entropy max): 'saner' (4.63s to compute)

Information Metrics Matrix (secret used for 'actual' columns: 'abled')
guess  prior_entropy_bits  expected_info_bits  n_buckets  max_bucket_frac actual_pattern  actual_info_bits  candidates_remaining
saner               8.886               5.668         94            0.129          .Y.G.             8.886                     1
slate               8.886               5.462         91            0.110          .YY.Y             6.564                     5
roate               8.886               5.403         81            0.129          ..Y.Y             6.078                     7
trace               8.886               5.343         84            0.154          ..Y.Y             6.564                     5
crane               8.886               5.315         85            0.169          ..Y.Y             6.564                     5
spare      